# Specsense Cell1


In [1]:
# ============================================================
#  SpecSense — Cell 1: Environment Setup & Document Parser
#  Google Colab · Free T4 GPU · No paid APIs
# ============================================================

# ── 1. Install all required packages in one shot ──────────────
!pip install -q \
    pdfplumber \
    pymupdf \
    python-docx \
    sentence-transformers \
    faiss-cpu \
    transformers \
    bitsandbytes \
    accelerate \
    gradio

print("✅ All packages installed successfully.")

# ── 2. Imports ────────────────────────────────────────────────
import os                        # path utilities
import math                      # ceiling division for DOCX page grouping
from typing import List, Dict    # type hints

# PDF parsing
import pdfplumber                # primary PDF text extractor

# DOCX parsing
from docx import Document as DocxDocument   # python-docx

# ── 3. DocumentParser class ───────────────────────────────────

class DocumentParser:
    """
    Unified parser for PDF and DOCX construction specification files.

    parse() returns a list of page dicts:
        [{"page": 1, "text": "..."}, {"page": 2, "text": "..."}, ...]

    Page numbers are preserved faithfully for PDF (actual page numbers
    from the file) and approximated for DOCX (every DOCX_PAGE_SIZE
    paragraphs = 1 logical page).
    """

    # Number of paragraphs treated as one "page" for DOCX files.
    # Adjust this constant if your specs have very short/long paragraphs.
    DOCX_PAGE_SIZE: int = 40

    # ── Public API ────────────────────────────────────────────
    def parse(self, file_path: str) -> List[Dict]:
        """
        Parse a .pdf or .docx file and return page-level text chunks.

        Parameters
        ----------
        file_path : str
            Absolute or relative path to the specification file.

        Returns
        -------
        List[Dict]
            Each element: {"page": int, "text": str}
            Pages are 1-indexed.

        Raises
        ------
        FileNotFoundError   – path does not exist on disk
        ValueError          – unsupported file extension
        RuntimeError        – document is corrupted or password-protected
        """

        # ── Validate path ─────────────────────────────────────
        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"File not found: '{file_path}'. "
                "Did you upload it correctly in the cell below?"
            )

        # ── Dispatch by extension ─────────────────────────────
        ext = os.path.splitext(file_path)[-1].lower()

        if ext == ".pdf":
            return self._parse_pdf(file_path)
        elif ext == ".docx":
            return self._parse_docx(file_path)
        else:
            raise ValueError(
                f"Unsupported file type '{ext}'. "
                "Only .pdf and .docx are accepted."
            )

    # ── Private helpers ───────────────────────────────────────

    def _parse_pdf(self, file_path: str) -> List[Dict]:
        """
        Extract text from a PDF using pdfplumber, one dict per page.

        pdfplumber is preferred over PyMuPDF for plain-text extraction
        because it handles construction-spec tables and column layouts
        more cleanly.  PyMuPDF (fitz) is reserved for the highlight-PDF
        step in a later cell.
        """
        pages: List[Dict] = []

        try:
            with pdfplumber.open(file_path) as pdf:

                # pdfplumber raises pdfminer.high_level.PDFPasswordIncorrect
                # for encrypted files — we catch the broad Exception below.
                for page_obj in pdf.pages:
                    page_number: int = page_obj.page_number   # 1-indexed
                    raw_text: str = page_obj.extract_text() or ""

                    # Strip excessive whitespace but keep paragraph breaks
                    cleaned_text: str = "\n".join(
                        line.strip()
                        for line in raw_text.splitlines()
                        if line.strip()          # drop blank lines
                    )

                    pages.append({
                        "page": page_number,
                        "text": cleaned_text,
                    })

        except FileNotFoundError:
            raise  # re-raise; already validated above, but just in case
        except Exception as exc:
            raise RuntimeError(
                f"Could not read PDF '{file_path}'. "
                "The file may be corrupted or password-protected.\n"
                f"Underlying error: {exc}"
            ) from exc

        return pages

    def _parse_docx(self, file_path: str) -> List[Dict]:
        """
        Extract text from a DOCX file using python-docx.

        DOCX has no native page-number concept in the XML, so we
        approximate pages by grouping every DOCX_PAGE_SIZE paragraphs
        together.  This gives the downstream RAG pipeline stable chunk
        IDs for traceability.
        """
        pages: List[Dict] = []

        try:
            doc = DocxDocument(file_path)
        except Exception as exc:
            raise RuntimeError(
                f"Could not open DOCX '{file_path}'. "
                "The file may be corrupted or use an unsupported format.\n"
                f"Underlying error: {exc}"
            ) from exc

        # Collect all non-empty paragraph texts
        all_paragraphs: List[str] = [
            para.text.strip()
            for para in doc.paragraphs
            if para.text.strip()   # ignore truly blank paragraphs
        ]

        if not all_paragraphs:
            # Return a single empty page rather than crashing
            return [{"page": 1, "text": ""}]

        # Group paragraphs into logical pages
        total_pages: int = math.ceil(len(all_paragraphs) / self.DOCX_PAGE_SIZE)

        for page_index in range(total_pages):
            start: int = page_index * self.DOCX_PAGE_SIZE
            end: int   = start + self.DOCX_PAGE_SIZE
            chunk: str = "\n".join(all_paragraphs[start:end])

            pages.append({
                "page": page_index + 1,   # 1-indexed
                "text": chunk,
            })

        return pages


# ── 4. Quick test: upload a file and sanity-check the parser ──

print("\n📂  Please upload your construction specification (PDF or DOCX)…")

# google.colab.files is only available inside a Colab runtime.
# If you're running locally, comment out the three lines below and
# set `file_path` manually, e.g.:  file_path = "spec.pdf"
try:
    from google.colab import files
    uploaded = files.upload()   # opens the Colab file-picker dialog

    # The upload widget returns a dict: {filename: bytes}
    file_name: str = list(uploaded.keys())[0]
except ImportError:
    # Local fallback
    import sys
    if len(sys.argv) > 1:
        file_name = sys.argv[1]
    else:
        file_name = "test_spec.pdf"
print(f"\n✅  Received file: '{file_name}'")

# ── Parse the uploaded file ───────────────────────────────────
parser = DocumentParser()

try:
    pages: List[Dict] = parser.parse(file_name)
except (FileNotFoundError, ValueError, RuntimeError) as err:
    print(f"❌  Parser error: {err}")
    pages = []

# ── Report results ────────────────────────────────────────────
if pages:
    total_pages: int = len(pages)
    print(f"\n📄  Total pages detected : {total_pages}")
    print("─" * 60)

    # Print the first three pages as a sanity check
    preview_count: int = min(3, total_pages)
    for i in range(preview_count):
        page_data = pages[i]
        # Truncate very long pages for readability
        preview_text: str = page_data["text"][:500]
        ellipsis: str = "…" if len(page_data["text"]) > 500 else ""

        print(f"\n┌── Page {page_data['page']} "
              f"({'chars: ' + str(len(page_data['text']))})")
        print(preview_text + ellipsis)
        print("└" + "─" * 59)

    print(f"\n✅  DocumentParser is working correctly — "
          f"{total_pages} page(s) extracted and ready for RAG indexing.")


✅ All packages installed successfully.

📂  Please upload your construction specification (PDF or DOCX)…


Saving Prescriptive Specifications_CPWD.pdf to Prescriptive Specifications_CPWD (2).pdf

✅  Received file: 'Prescriptive Specifications_CPWD (2).pdf'

📄  Total pages detected : 107
────────────────────────────────────────────────────────────

┌── Page 1 (chars: 3372)
4.0 CONCRETE WORK
4.1. MATERIAL
Water, cement, fine aggregate or sand, surkhi, and fly ash shall be as specified in Chapter 3.0 –
Mortar.
4.1.1 Coarse Aggregate
4.1.1.1 General: Aggregate most of which is retained on 4.75 mm IS Sieve and contains only as much
fine material as is permitted in IS 383 for various sizes and grading is known as coarse aggregate.
Coarse aggregate shall be specified as stone aggregate, gravel or brick aggregate and it shall be
obtained from approved/ authorized sources…
└───────────────────────────────────────────────────────────

┌── Page 2 (chars: 1975)
TABLE 4.1
Graded Stone Aggregate or Gravel
IS Sieve Percentage passing (by weight) for nominal size of
Designation 40 mm 20 mm 16 mm 12.5 mm
80

# Specsense Cell2


In [2]:
# ============================================================
#  SpecSense — Cell 2: Semantic Chunker + FAISS Index Builder
#  Depends on: Cell 1 (DocumentParser, `pages` variable)
# ============================================================
import re                             # regex for section-label detection
import math                           # ceiling division
from typing import List, Dict, Tuple  # type hints

import numpy as np                    # vector normalisation
import faiss                          # vector similarity search
from sentence_transformers import SentenceTransformer  # free embedding model

# ══════════════════════════════════════════════════════════════
#  CLASS 1 — SemanticChunker
# ══════════════════════════════════════════════════════════════

class SemanticChunker:
    """
    Splits page-level text into overlapping word-window chunks and
    attaches a construction-domain section label to every chunk.

    Why overlapping windows?
    ────────────────────────
    A key sentence about curing temperatures might sit at the boundary
    of two non-overlapping chunks and be missed by the retriever.
    Overlapping windows ensure boundary context is always captured.
    """

    # ── Keyword map used by detect_section_label ──────────────
    # Each entry: label -> list of lowercase keyword fragments.
    # Fragments are matched as substrings (no word-boundary required)
    # so "vibrat" matches "vibration", "vibrator", "vibrating", etc.
    _KEYWORD_MAP: Dict[str, List[str]] = {
        "MATERIALS":  [
            "aggregate", "cement", "water", "admixture",
            "fly ash", "flyash", "sand", "gravel", "coarse",
            "fine aggregate", "supplementary", "additive",
        ],
        "PROCEDURE":  [
            "mixing", "placing", "placement", "compaction",
            "curing", "batching", "pouring", "pour", "vibrat",
            "casting", "finishing", "levelling", "leveling",
            "consolidat", "transportation", "discharge",
        ],
        "EQUIPMENT":  [
            "plant", "mixer", "vibrator", "pump", "transit",
            "crane", "formwork", "shuttering", "truck", "chute",
            "conveyor", "batch", "hopper", "transit mixer",
        ],
        "STANDARDS":  [
            "is:", "is 383", "is 456", "is 10262", "is 4926",
            "clause", " code", "specification", "conform",
            "standard", "astm", "bs ", "en ", "aci ", "irc",
        ],
        "PERSONNEL":  [
            "engineer-in-charge", "engineer in charge",
            "supervisor", "inspector", "quality", " qc ",
            "officer", "foreman", "technician", "site engineer",
            "project manager",
        ],
        # GENERAL is the fallback — applied when nothing else matches
    }

    # ── Public API ────────────────────────────────────────────

    def chunk(
        self,
        pages: List[Dict],
        chunk_size: int = 400,
        overlap: int = 80,
    ) -> List[Dict]:
        """
        Split every page's text into overlapping word-window chunks.

        Parameters
        ----------
        pages      : output of DocumentParser.parse()  (list of page dicts)
        chunk_size : target window size in words
        overlap    : number of words shared between consecutive chunks
                     on the same page

        Returns
        -------
        List[Dict] with keys:
            chunk_id      – globally unique int (0-indexed)
            page          – source page number (from the original document)
            text          – chunk text string
            char_start    – character offset of the first word in the page text
            char_end      – character offset after the last word in the page text
            section_label – domain label string
        """
        if chunk_size <= overlap:
            raise ValueError(
                f"chunk_size ({chunk_size}) must be greater than overlap ({overlap})."
            )

        all_chunks: List[Dict] = []
        global_chunk_id: int = 0
        step: int = chunk_size - overlap      # how far we advance each window

        for page_dict in pages:
            page_num: int  = page_dict["page"]
            full_text: str = page_dict["text"]

            # ── Tokenise by whitespace ─────────────────────────
            # We keep track of character offsets so we can highlight
            # exact spans back in the PDF in a later cell.
            words: List[str] = full_text.split()
            if not words:
                # Empty page — still emit one stub chunk so page IDs are traceable
                all_chunks.append({
                    "chunk_id":     global_chunk_id,
                    "page":         page_num,
                    "text":         "",
                    "char_start":   0,
                    "char_end":     0,
                    "section_label": "GENERAL",
                })
                global_chunk_id += 1
                continue

            # Build char-offset index: word_offsets[i] = start char of words[i]
            word_offsets: List[int] = self._build_word_offsets(full_text, words)

            # ── Slide window over words ────────────────────────
            num_words: int = len(words)
            start_idx: int = 0

            while start_idx < num_words:
                end_idx: int = min(start_idx + chunk_size, num_words)

                chunk_words: List[str] = words[start_idx:end_idx]
                chunk_text: str        = " ".join(chunk_words)

                # Character offsets within the page text
                char_start: int = word_offsets[start_idx]
                # end offset = start of last word + its length
                char_end: int   = word_offsets[end_idx - 1] + len(words[end_idx - 1])

                # Detect which construction domain this chunk belongs to
                label: str = self.detect_section_label(chunk_text)

                all_chunks.append({
                    "chunk_id":     global_chunk_id,
                    "page":         page_num,
                    "text":         chunk_text,
                    "char_start":   char_start,
                    "char_end":     char_end,
                    "section_label": label,
                })

                global_chunk_id += 1

                # Advance window; stop if we've covered all words
                if end_idx == num_words:
                    break
                start_idx += step

        return all_chunks

    def detect_section_label(self, chunk_text: str) -> str:
        """
        Heuristically classify a text chunk into a construction domain.

        Strategy: lowercase the text, then count keyword hits for each
        category.  The category with the most hits wins.  Ties are
        broken by the order in _KEYWORD_MAP (MATERIALS > PROCEDURE > …).
        Returns "GENERAL" if no keywords match.

        Parameters
        ----------
        chunk_text : raw text string of a single chunk

        Returns
        -------
        str — one of: MATERIALS | PROCEDURE | EQUIPMENT |
                       STANDARDS | PERSONNEL | GENERAL
        """
        lower_text: str = chunk_text.lower()

        best_label: str = "GENERAL"
        best_score: int = 0

        for label, keywords in self._KEYWORD_MAP.items():
            score: int = sum(
                1 for kw in keywords
                if kw in lower_text          # simple substring match
            )
            if score > best_score:
                best_score = score
                best_label = label

        return best_label

    # ── Private helpers ───────────────────────────────────────

    @staticmethod
    def _build_word_offsets(full_text: str, words: List[str]) -> List[int]:
        """
        Return the character-start index of every word inside full_text.

        We scan forward through full_text so that multi-space gaps and
        newlines are accounted for correctly.
        """
        offsets: List[int] = []
        search_start: int  = 0

        for word in words:
            # find() is O(n) but text is short enough that this is fine
            idx: int = full_text.find(word, search_start)
            offsets.append(idx)
            search_start = idx + len(word)

        return offsets


# ══════════════════════════════════════════════════════════════
#  CLASS 2 — FAISSIndexBuilder
# ══════════════════════════════════════════════════════════════

class FAISSIndexBuilder:
    """
    Embeds chunk texts with a lightweight sentence-transformer model
    and stores them in a FAISS IndexFlatIP for cosine-similarity search.

    Model choice: "all-MiniLM-L6-v2"
    ──────────────────────────────────
    • 22 M parameters  → fits easily in Colab free RAM alongside Mistral-7B
    • 384-dimensional embeddings  → FAISS index is tiny
    • Strong performance on sentence retrieval benchmarks
    • Downloaded once; cached by HuggingFace Hub automatically
    """

    # Embedding model identifier (HuggingFace Hub)
    MODEL_NAME: str = "all-MiniLM-L6-v2"

    def __init__(self) -> None:
        print(f"⏳  Loading embedding model '{self.MODEL_NAME}'…")
        # SentenceTransformer auto-selects GPU if available (free T4 in Colab)
        self._model: SentenceTransformer = SentenceTransformer(self.MODEL_NAME)
        print(f"✅  Embedding model loaded "
              f"(dim={self._model.get_embedding_dimension()}).")

    # ── Public API ────────────────────────────────────────────

    def build(
        self,
        chunks: List[Dict],
    ) -> Tuple[faiss.Index, List[Dict]]:
        """
        Embed all chunks and build a FAISS cosine-similarity index.

        Parameters
        ----------
        chunks : list of chunk dicts from SemanticChunker.chunk()

        Returns
        -------
        (faiss_index, chunks)
            faiss_index – IndexFlatIP ready for .search()
            chunks      – same list passed in (returned for convenience
                          so caller can store both in one tuple)
        """
        print(f"🔢  Embedding {len(chunks)} chunks…")

        # Extract raw texts; FAISS needs a clean list of strings
        texts: List[str] = [c["text"] for c in chunks]

        # Encode in one batched call — SentenceTransformer handles batching
        # convert_to_numpy=True gives us a float32 ndarray directly
        embeddings: np.ndarray = self._model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            batch_size=64,           # safe default for Colab T4
        )                            # shape: (num_chunks, embedding_dim)

        # ── Normalise to unit length ───────────────────────────
        # With L2-normalised vectors, inner-product = cosine similarity.
        # This lets us use the fast IndexFlatIP without a separate
        # cosine-distance index type.
        faiss.normalize_L2(embeddings)

        # ── Build index ────────────────────────────────────────
        embedding_dim: int = embeddings.shape[1]
        index: faiss.Index = faiss.IndexFlatIP(embedding_dim)
        index.add(embeddings)        # add all vectors at once

        print(f"✅  FAISS index built — {index.ntotal} vectors, dim={embedding_dim}.")
        return index, chunks

    def search(
        self,
        query: str,
        faiss_index: faiss.Index,
        chunks: List[Dict],
        top_k: int = 5,
    ) -> List[Dict]:
        """
        Retrieve the top-k most semantically similar chunks for a query.

        Parameters
        ----------
        query       : natural-language question or section heading
        faiss_index : index returned by build()
        chunks      : chunk list returned by build()
        top_k       : number of results to return

        Returns
        -------
        List[Dict] — copies of the matched chunk dicts, each with an
        additional key "similarity_score" (float, range 0–1 after
        normalisation; higher = more similar).
        """
        # ── Embed and normalise the query ──────────────────────
        query_vec: np.ndarray = self._model.encode(
            [query],
            convert_to_numpy=True,
        )                            # shape: (1, embedding_dim)
        faiss.normalize_L2(query_vec)

        # ── Search ─────────────────────────────────────────────
        # FAISS returns two arrays of shape (1, top_k):
        #   scores  – inner-product values (≈ cosine similarity after norm)
        #   indices – positions in the index (= chunk list positions)
        scores: np.ndarray
        indices: np.ndarray
        scores, indices = faiss_index.search(query_vec, top_k)

        # ── Build result list ──────────────────────────────────
        results: List[Dict] = []
        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0])
        ):
            if idx == -1:
                # FAISS returns -1 when fewer than top_k results exist
                continue

            # Make a shallow copy so we don't mutate the original chunk
            result: Dict = dict(chunks[idx])
            result["similarity_score"] = float(score)
            results.append(result)

        return results


# ══════════════════════════════════════════════════════════════
#  PIPELINE — Run on `pages` from Cell 1
# ══════════════════════════════════════════════════════════════

# ── Step A: Chunk the parsed pages ────────────────────────────
print("\n" + "═" * 60)
print("  STEP A — Semantic Chunking")
print("═" * 60)

chunker  = SemanticChunker()
chunks: List[Dict] = chunker.chunk(
    pages,           # `pages` variable populated by Cell 1's test harness
    chunk_size=400,  # words per window  (tune down if GPU OOM on large docs)
    overlap=80,      # words shared between consecutive windows
)

total_chunks: int = len(chunks)
print(f"\n📦  Total chunks created : {total_chunks}")

# ── Section label distribution ─────────────────────────────────
print("\n📊  Section label distribution:")
print("─" * 40)

from collections import Counter
label_counts: Counter = Counter(c["section_label"] for c in chunks)

# Sort by count descending for readability
for label, count in label_counts.most_common():
    bar: str = "█" * int(count / max(label_counts.values()) * 30)
    pct: float = count / total_chunks * 100
    print(f"  {label:<12} {count:>4} chunks  ({pct:5.1f}%)  {bar}")

# ── Step B: Build FAISS index ──────────────────────────────────
print("\n" + "═" * 60)
print("  STEP B — FAISS Index Building")
print("═" * 60)

builder = FAISSIndexBuilder()
faiss_index, chunks = builder.build(chunks)

# ── Step C: Smoke-test retrieval with a sample query ───────────
print("\n" + "═" * 60)
print("  STEP C — Retrieval Smoke Test")
print("═" * 60)

SAMPLE_QUERY: str = "What are the curing requirements for concrete?"
print(f"\n🔍  Query: \"{SAMPLE_QUERY}\"")
print("─" * 60)

results: List[Dict] = builder.search(
    query       = SAMPLE_QUERY,
    faiss_index = faiss_index,
    chunks      = chunks,
    top_k       = 3,
)

for rank, res in enumerate(results, start=1):
    preview: str = res["text"][:300].replace("\n", " ")
    ellipsis: str = "…" if len(res["text"]) > 300 else ""
    print(
        f"\n  [{rank}]  page={res['page']}  "
        f"label={res['section_label']}  "
        f"score={res['similarity_score']:.4f}\n"
        f"      {preview}{ellipsis}"
    )

print("\n✅  Cell 2 complete — `faiss_index` and `chunks` ready for Cell 3 (LLM generation).")



════════════════════════════════════════════════════════════
  STEP A — Semantic Chunking
════════════════════════════════════════════════════════════

📦  Total chunks created : 179

📊  Section label distribution:
────────────────────────────────────────
  MATERIALS      73 chunks  ( 40.8%)  ██████████████████████████████
  STANDARDS      70 chunks  ( 39.1%)  ████████████████████████████
  PROCEDURE      19 chunks  ( 10.6%)  ███████
  EQUIPMENT      12 chunks  (  6.7%)  ████
  GENERAL         4 chunks  (  2.2%)  █
  PERSONNEL       1 chunks  (  0.6%)  

════════════════════════════════════════════════════════════
  STEP B — FAISS Index Building
════════════════════════════════════════════════════════════
⏳  Loading embedding model 'all-MiniLM-L6-v2'…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅  Embedding model loaded (dim=384).
🔢  Embedding 179 chunks…


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅  FAISS index built — 179 vectors, dim=384.

════════════════════════════════════════════════════════════
  STEP C — Retrieval Smoke Test
════════════════════════════════════════════════════════════

🔍  Query: "What are the curing requirements for concrete?"
────────────────────────────────────────────────────────────

  [1]  page=60  label=MATERIALS  score=0.6730
      felt or any such material and provision of copper plate, etc. shall be paid for separately in running metre. The measurement shall be taken two places of decimal stating the depth and width of joint. 5.4.6 Curing After the concrete has begun to harden i.e. about 1 to 2 hours after its laying, it sha…

  [2]  page=83  label=MATERIALS  score=0.6470
      chemicaladmixtures and super plasticizers with each set OPC, fly ash and /or PPC received from differentsources shall be ensured by trials. (v) In environment subjected to aggressive chloride or sulphate attack in particular, use of fly ashadmixed or PPC based concrete i

# Specsense Cell3


In [3]:
# ============================================================
#  SpecSense — Cell 3: LLM Initialization & Helper Functions
#  Google Colab · Free T4 GPU · No paid APIs
# ============================================================

import json
import re
from typing import Dict, Optional

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# ── 1. Check GPU availability ─────────────────────────────────
print("═" * 60)
print("  STEP 1 — Checking GPU")
print("═" * 60)

if not torch.cuda.is_available():
    print("❌ GPU is NOT available! You are running on CPU.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    # We set a dummy device just in case, though 4-bit quant needs GPU.
    device = torch.device("cpu")
else:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected: {gpu_name} (VRAM: {vram_gb:.1f} GB)")

# ── 2. Load Model and Tokenizer ───────────────────────────────
print("\n" + "═" * 60)
print("  STEP 2 — Loading Mistral-7B-Instruct-v0.3 (4-bit)")
print("═" * 60)

MODEL_ID: str = "mistralai/Mistral-7B-Instruct-v0.3"

# Configure 4-bit quantization to fit the 7B model within 16GB VRAM
# This dramatically reduces memory footprint while maintaining decent accuracy.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"⏳ Downloading and loading model weights for '{MODEL_ID}'...")
print("   (This might take a few minutes the first time to download ~4-5GB)")

try:
    # device_map="auto" lets accelerate handle distributing model layers
    # across available GPUs (or offloading if needed, but T4 is enough).
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print("✅ Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"❌ Failed to load model. Error: {e}")
    print("   Did you accept the Mistral terms on HuggingFace Hub?")
    print("   If gated, you may need to run `huggingface-cli login` first.")
    raise

# ── 3. Helper: llm_generate ───────────────────────────────────

def llm_generate(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Generate text from the loaded Mistral model given a prompt.
    Formats the prompt using Mistral's instruction template.
    """
    # 1. Format prompt for Mistral v0.3 Instruct
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"

    # 2. Tokenize and move to correct device (typically GPU)
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    # 3. Generate response
    # do_sample=False -> greedy decoding (more deterministic, good for extraction)
    # repetition_penalty=1.1 -> slight penalty to prevent repetitive loops
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,        # Has no effect if do_sample=False, but explicit
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id # Suppress warning
        )

    # 4. Decode
    # The output includes the input prompt. We only want the new tokens.
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output_ids[0][input_length:]

    result: str = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return result.strip()

# ── 4. Helper: llm_extract_json ───────────────────────────────

def llm_extract_json(prompt: str, max_new_tokens: int = 1536) -> List[Dict]:
    """
    Calls the LLM and attempts to extract and parse all JSON dictionaries
    from its response. Resilient to markdown formatting or conversational filler.
    Always returns a list of dictionaries (may be empty).
    """
    # Ask the LLM to generate the output
    raw_response: str = llm_generate(prompt, max_new_tokens=max_new_tokens)

    if not raw_response:
        print("⚠️ Warning: LLM returned an empty response.")
        return []

    # Strategy 1: Find all JSON blocks using a decoder-based search
    # This is more robust than regex as it handles nested braces.
    results = []
    pos = 0
    while pos < len(raw_response):
        # Find start of a JSON object or array
        match = re.search(r'[\{\[]', raw_response[pos:])
        if not match:
            break

        start_index = pos + match.start()
        try:
            decoder = json.JSONDecoder()
            obj, end_pos = decoder.raw_decode(raw_response[start_index:])
            if isinstance(obj, dict):
                results.append(obj)
            elif isinstance(obj, list):
                for item in obj:
                    if isinstance(item, dict):
                        results.append(item)
            pos = start_index + end_pos
        except json.JSONDecodeError:
            pos = start_index + 1

    if not results:
        # Fallback Strategy 1: Greedy regex for single block (legacy)
        match = re.search(r'(\{.*\})', raw_response, re.DOTALL)
        if match:
            json_str = match.group(1)
            try:
                # Try to fix potential multiple objects separated by commas by wrapping in []
                fixed_str = f"[{json_str}]"
                objs = json.loads(fixed_str)
                if isinstance(objs, list):
                    results.extend([o for o in objs if isinstance(o, dict)])
            except json.JSONDecodeError:
                pass # Try repair below

        # Fallback Strategy 2: Attempt to repair truncated JSON
        # If the LLM cut off, we might have an unclosed { or [
        if not results and '{' in raw_response:
            # Find the last start of a JSON object
            last_start = raw_response.rfind('{')
            potential_json = raw_response[last_start:]
            # Try to close it
            for closure in ["}", '"}', '"]}', '"]}', '"]}']:
                try:
                    obj = json.loads(potential_json + closure)
                    if isinstance(obj, dict):
                        results.append(obj)
                        break
                except: continue

            # If still nothing, try more aggressive repair (closing all open structures)
            if not results:
                repaired = raw_response
                # Find the first {
                first_start = raw_response.find('{')
                if first_start != -1:
                    repaired = raw_response[first_start:]
                    # Simple heuristic: add enough closures
                    # Count opens and closes
                    opens_brace = repaired.count('{')
                    closes_brace = repaired.count('}')
                    opens_bracket = repaired.count('[')
                    closes_bracket = repaired.count(']')

                    # Add missing closes
                    # We also need to close the last string if it's open
                    if repaired.count('"') % 2 != 0:
                        repaired += '"'

                    repaired += ']' * (opens_bracket - closes_bracket)
                    repaired += '}' * (opens_brace - closes_brace)

                    try:
                        obj = json.loads(repaired)
                        if isinstance(obj, dict): results.append(obj)
                        elif isinstance(obj, list): results.extend([o for o in obj if isinstance(o, dict)])
                    except: pass

    if not results:
        print("⚠️ Warning: Could not find any valid JSON block in LLM response.")

    return results

# ── 5. Quick Test ─────────────────────────────────────────────
print("\n" + "═" * 60)
print("  STEP 3 — Testing LLM Generation")
print("═" * 60)

test_prompt = "What is cement? Give a brief 2-sentence explanation."
print(f"❓ Prompt: {test_prompt}")
print("⏳ Generating response (compiling kernels for the first time, may be slow)...\n")

try:
    test_response = llm_generate(test_prompt, max_new_tokens=100)
    print(f"💡 Response: {test_response}")
    print("\n✅ LLM functions are ready to use!")
except Exception as e:
    print(f"❌ LLM generation failed: {e}")


════════════════════════════════════════════════════════════
  STEP 1 — Checking GPU
════════════════════════════════════════════════════════════
✅ GPU detected: Tesla T4 (VRAM: 14.6 GB)

════════════════════════════════════════════════════════════
  STEP 2 — Loading Mistral-7B-Instruct-v0.3 (4-bit)
════════════════════════════════════════════════════════════
⏳ Downloading and loading model weights for 'mistralai/Mistral-7B-Instruct-v0.3'...
   (This might take a few minutes the first time to download ~4-5GB)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model and tokenizer loaded successfully.

════════════════════════════════════════════════════════════
  STEP 3 — Testing LLM Generation
════════════════════════════════════════════════════════════
❓ Prompt: What is cement? Give a brief 2-sentence explanation.
⏳ Generating response (compiling kernels for the first time, may be slow)...

💡 Response: Cement is a fine, grey powdery material that, when mixed with water and aggregates like sand and gravel, creates a hard, durable substance used primarily in construction for making mortar, concrete, and various other building materials. It's produced by heating limestone and clay or other raw materials to high temperatures in a kiln, resulting in a clinker which is then ground into the familiar cement powder.

✅ LLM functions are ready to use!


# Specsense Cell4


In [4]:
# ============================================================
#  SpecSense — Cell 4: Extraction Agents
#  Depends on: Cell 2 (FAISSIndexBuilder, chunks), Cell 3 (LLM Helpers)
# ============================================================

from typing import List, Dict, Any

# ── 1. Extraction Agent Class ─────────────────────────────────
print("═" * 60)
print("  STEP 1 — Defining ExtractionAgent Class")
print("═" * 60)

class ExtractionAgent:
    """
    An agent dedicated to extracting specific information from a
    document section using FAISS retrieval and Mistral-7B JSON extraction.
    """

    def __init__(
        self,
        agent_name: str,
        target_section: str,
        extraction_queries: List[str]
    ) -> None:
        self.agent_name = agent_name
        self.target_section = target_section
        self.extraction_queries = extraction_queries

    def _build_context(self, search_results: List[Dict]) -> str:
        """
        Formats FAISS search results into a single context string for the LLM.
        """
        context_parts = []
        for rank, chunk in enumerate(search_results, start=1):
            # Include page number to help LLM cite the source
            context_parts.append(
                f"[Document Page: {chunk['page']}]\n{chunk['text']}"
            )
        return "\n\n---\n\n".join(context_parts)

    def extract(
        self,
        faiss_index: Any,
        chunks: List[Dict],
        builder: Any,
        top_k: int = 6
    ) -> List[Dict]:
        """
        Executes extraction queries against the document.

        Parameters
        ----------
        faiss_index : faiss.Index built in Cell 2
        chunks      : List of all chunk dicts built in Cell 2
        builder     : FAISSIndexBuilder instance to perform the search
        top_k       : Number of chunks to retrieve per query

        Returns
        -------
        List of extraction result dicts.
        """
        print(f"\n🚀 Starting {self.agent_name} ({len(self.extraction_queries)} queries)")
        results: List[Dict] = []

        for query in self.extraction_queries:
            # 1. Search FAISS for relevant chunks
            search_results: List[Dict] = builder.search(
                query=query,
                faiss_index=faiss_index,
                chunks=chunks,
                top_k=top_k
            )

            # 2. Filter / Prioritize chunks based on target_section
            # Note: We keep chunks that match the section, or high similarity general chunks
            # For simplicity, we'll just sort the retrieved chunks, bringing target section to the top.
            search_results.sort(
                key=lambda x: (x.get("section_label") == self.target_section, x.get("similarity_score", 0)),
                reverse=True
            )

            context: str = self._build_context(search_results)

            # 3. Build strict extraction prompt
            prompt: str = f"""You are a construction specification analyst. Read the following passages from a
construction specification document and extract ONLY information that is explicitly
stated in the text. Do NOT add any information from your general knowledge.
If the information is not present in the passages, respond with: {{"value": "NOT_FOUND"}}

PASSAGES:
{context}

QUESTION: {query}

Respond ONLY with valid JSON in this exact format:
{{
  "value": "extracted info (summarize concisely if there are many requirements)",
  "source_page": <page number as integer>,
  "source_clause": "the clause or section reference e.g. 4.1.1.3",
  "verbatim_snippet": "the exact sentence or phrase from the text that supports this"
}}"""

            # 4. Call LLM
            print(f"  🔍 Query: {query}")
            extracted_items = llm_extract_json(prompt)

            # 5. Process Results
            if extracted_items:
                for item in extracted_items:
                    results.append({
                        "agent": self.agent_name,
                        "field": query,
                        "value": item.get("value", "NOT_FOUND"),
                        "source_page": item.get("source_page", -1),
                        "source_clause": item.get("source_clause", ""),
                        "verbatim_snippet": item.get("verbatim_snippet", ""),
                        "confidence": 0.9 if item.get("value") != "NOT_FOUND" else 0.0
                    })
            else:
                results.append({
                    "agent": self.agent_name,
                    "field": query,
                    "value": "NOT_FOUND",
                    "source_page": -1,
                    "source_clause": "",
                    "verbatim_snippet": "",
                    "confidence": 0.0
                })

        return results

# ── 2. Instantiate Agents ─────────────────────────────────────
print("\n" + "═" * 60)
print("  STEP 2 — Initialising Specialised Agents")
print("═" * 60)

# AGENT 1 — Materials
materials_agent = ExtractionAgent(
    agent_name="Materials Agent",
    target_section="MATERIALS",
    extraction_queries=[
        "What type of cement should be used and what are its requirements?",
        "What are the requirements for coarse aggregate including size and grading?",
        "What are the requirements for fine aggregate or sand?",
        "What is the maximum water-cement ratio allowed?",
        "What admixtures or fly ash can be used in concrete?",
        "What are the deleterious material limits for aggregates?"
    ]
)

# AGENT 2 — Procedure
procedure_agent = ExtractionAgent(
    agent_name="Procedure Agent",
    target_section="PROCEDURE",
    extraction_queries=[
        "What is the procedure for batching and mixing concrete?",
        "What are the requirements for placing and pouring concrete?",
        "How should concrete be compacted or vibrated?",
        "What are the curing requirements and duration for concrete?",
        "What are the minimum concrete grades or mix proportions specified?",
        "What are the requirements for formwork?"
    ]
)

# AGENT 3 — Equipment
equipment_agent = ExtractionAgent(
    agent_name="Equipment Agent",
    target_section="EQUIPMENT",
    extraction_queries=[
        "What equipment or machinery is required for concrete batching and mixing?",
        "What type of vibrators or compaction equipment is specified?",
        "What transport equipment is needed for concrete delivery?",
        "What testing equipment is required for quality control?"
    ]
)

# AGENT 4 — Standards
standards_agent = ExtractionAgent(
    agent_name="Standards Agent",
    target_section="STANDARDS",
    extraction_queries=[
        "What IS codes or Indian Standards are referenced?",
        "What CPWD specifications or clauses are referenced?",
        "What testing standards must concrete comply with?",
        "What are the acceptance criteria or tolerances specified?"
    ]
)

# AGENT 5 — Personnel
personnel_agent = ExtractionAgent(
    agent_name="Personnel Agent",
    target_section="PERSONNEL",
    extraction_queries=[
        "Who is the Engineer-in-Charge and what are their responsibilities?",
        "What quality control personnel or inspectors are required?",
        "What approvals or sign-offs are required before concreting?"
    ]
)

agents: List[ExtractionAgent] = [
    materials_agent,
    procedure_agent,
    equipment_agent,
    standards_agent,
    personnel_agent
]

# ── 3. Run Pipeline ───────────────────────────────────────────
print("\n" + "═" * 60)
print("  STEP 3 — Running Extraction Pipeline")
print("═" * 60)

all_extractions: List[Dict] = []

for agent in agents:
    # Assuming `faiss_index`, `chunks`, and `builder` are available from Cell 2
    # This might take a while depending on doc size and number of queries.
    agent_results = agent.extract(faiss_index, chunks, builder, top_k=6)
    all_extractions.extend(agent_results)

# ── 4. Print Summary ──────────────────────────────────────────
print("\n" + "═" * 60)
print("  EXTRACTION SUMMARY")
print("═" * 60)

# Group results by agent
summary_counts: Dict[str, Dict[str, int]] = {
    agent.agent_name: {"found": 0, "not_found": 0, "error": 0}
    for agent in agents
}

for result in all_extractions:
    agent_name = result["agent"]
    val = result["value"]

    if val == "NOT_FOUND":
        summary_counts[agent_name]["not_found"] += 1
    elif val == "JSON_PARSE_ERROR":
        summary_counts[agent_name]["error"] += 1
    else:
        summary_counts[agent_name]["found"] += 1

# Print the breakdown
for agent_name, counts in summary_counts.items():
    print(f"\n🤖 {agent_name}:")
    print(f"   ✅ Found    : {counts['found']}")
    print(f"   ❌ Not Found: {counts['not_found']}")
    if counts['error'] > 0:
        print(f"   ⚠️ Errors   : {counts['error']}")

# Show flags for any missing information
print("\n" + "─" * 60)
print("⚠️ MISSING INFORMATION (NOT_FOUND):")
for result in all_extractions:
    if result["value"] == "NOT_FOUND":
        print(f" - [{result['agent']}] {result['field']}")

print("\n✅ Cell 4 complete. Extractions stored in `all_extractions`.")


════════════════════════════════════════════════════════════
  STEP 1 — Defining ExtractionAgent Class
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 2 — Initialising Specialised Agents
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 3 — Running Extraction Pipeline
════════════════════════════════════════════════════════════

🚀 Starting Materials Agent (6 queries)
  🔍 Query: What type of cement should be used and what are its requirements?
  🔍 Query: What are the requirements for coarse aggregate including size and grading?
  🔍 Query: What are the requirements for fine aggregate or sand?
  🔍 Query: What is the maximum water-cement ratio allowed?
  🔍 Query: What admixtures or fly ash can be used in concrete?
  🔍 Query: What are the deleterious material limits for aggregates?

🚀 Starting Procedure Agent (6 queries)
  🔍 Query:

# Specsense Cell5


In [5]:
# ============================================================
#  SpecSense — Cell 5: Grounding & Hallucination Validator
#  Depends on: Cell 1 (pages), Cell 4 (all_extractions)
# ============================================================

import difflib
from typing import List, Dict, Any

# ── 1. Grounding Validator Class ──────────────────────────────
print("═" * 60)
print("  STEP 1 — Defining GroundingValidator Class")
print("═" * 60)

class GroundingValidator:
    """
    Validates LLM-extracted facts by cross-referencing their
    reported 'verbatim_snippet' back to the source text.

    This acts as our core anti-hallucination mechanism.
    """

    def validate(self, extractions: List[Dict], pages: List[Dict]) -> List[Dict]:
        """
        Check if the verbatim snippet actually exists in the original document page.
        """
        validated_extractions: List[Dict] = []

        # Build a fast lookup dictionary for page texts
        page_lookup: Dict[int, str] = {p["page"]: p["text"] for p in pages}

        for ext in extractions:
            # Create a copy so we don't mutate the original dictionary
            result = dict(ext)

            # Skip validation for items where no information was found or parsing failed
            if result.get("value") in ("NOT_FOUND", "JSON_PARSE_ERROR"):
                result["grounded"] = False
                result["match_score"] = 0.0
                result["status"] = "SKIPPED"
                validated_extractions.append(result)
                continue

            snippet_raw = result.get("verbatim_snippet", "")
            if isinstance(snippet_raw, list):
                snippet: str = " ".join([str(s) for s in snippet_raw]).strip()
            else:
                snippet: str = str(snippet_raw).strip()

            source_page_raw = result.get("source_page", -1)

            # Handle source_page being a list, string, or other type
            if isinstance(source_page_raw, list):
                source_page = source_page_raw[0] if source_page_raw else -1
            else:
                source_page = source_page_raw

            # Ensure source_page is an integer for dictionary lookup
            try:
                source_page = int(source_page)
            except (ValueError, TypeError):
                source_page = -1

            # If the LLM didn't provide a snippet or page, it's immediately unverified
            if not snippet or source_page not in page_lookup:
                result["grounded"] = False
                result["match_score"] = 0.0
                result["status"] = "UNVERIFIED"
                validated_extractions.append(result)
                continue

            page_text: str = page_lookup[source_page]

            # Try 1: Exact substring match (case-insensitive)
            if snippet.lower() in page_text.lower():
                result["grounded"] = True
                result["match_score"] = 1.0
                result["status"] = "VERIFIED"
                validated_extractions.append(result)
                continue

            # Try 2: Fuzzy match (handles minor whitespace or punctuation changes)
            # SequenceMatcher gets slow on huge texts, so we compare line-by-line
            # or chunk-by-chunk for better performance, but for Colab scale against
            # single pages, direct comparison is usually fine.
            # We use a sliding window over the page text to find the best match.
            best_ratio: float = 0.0
            snippet_len: int = len(snippet)

            # Quick heuristic: if the page is extremely long, difflib might be slow.
            # We just do a standard SequenceMatcher against the whole page text.
            # But since we want to find a matching *snippet*, we can split the page into sentences/lines.
            lines = [line.strip() for line in page_text.split('\n') if line.strip()]

            for line in lines:
                ratio = difflib.SequenceMatcher(None, snippet.lower(), line.lower()).ratio()
                if ratio > best_ratio:
                    best_ratio = ratio

            if best_ratio > 0.75:
                result["grounded"] = True
                result["match_score"] = best_ratio
                result["status"] = "FUZZY_MATCH"
            else:
                result["grounded"] = False
                result["match_score"] = best_ratio
                result["status"] = "UNVERIFIED"

            validated_extractions.append(result)

        return validated_extractions

    def compute_grounding_score(self, validated_extractions: List[Dict]) -> Dict[str, Any]:
        """
        Computes the overall accuracy / grounding score for the hackathon submission.
        """
        # Only consider facts that were actually extracted (ignore SKIPPED)
        facts = [ext for ext in validated_extractions if ext.get("status") != "SKIPPED"]

        total_facts = len(facts)
        verified = sum(1 for ext in facts if ext.get("status") == "VERIFIED")
        fuzzy = sum(1 for ext in facts if ext.get("status") == "FUZZY_MATCH")
        unverified = sum(1 for ext in facts if ext.get("status") == "UNVERIFIED")

        grounding_score = ((verified + fuzzy) / total_facts * 100) if total_facts > 0 else 0.0

        # Calculate by agent
        by_agent: Dict[str, Dict[str, Any]] = {}
        for ext in facts:
            agent = ext["agent"]
            if agent not in by_agent:
                by_agent[agent] = {"total": 0, "grounded": 0}

            by_agent[agent]["total"] += 1
            if ext.get("status") in ("VERIFIED", "FUZZY_MATCH"):
                by_agent[agent]["grounded"] += 1

        # Finalize by_agent scores
        for agent, stats in by_agent.items():
            t = stats["total"]
            stats["score"] = (stats["grounded"] / t * 100) if t > 0 else 0.0

        return {
            "total_facts": total_facts,
            "verified": verified,
            "fuzzy_match": fuzzy,
            "unverified": unverified,
            "grounding_score": grounding_score,
            "by_agent": by_agent
        }

    def get_verified_extractions(self, validated_extractions: List[Dict]) -> List[Dict]:
        """
        Returns only the extractions that passed grounding validation.
        """
        return [
            ext for ext in validated_extractions
            if ext.get("status") in ("VERIFIED", "FUZZY_MATCH")
        ]

# ── 2. Run Pipeline ───────────────────────────────────────────
print("\n" + "═" * 60)
print("  STEP 2 — Running Grounding Validation")
print("═" * 60)

validator = GroundingValidator()

# We assume `all_extractions` and `pages` are loaded from Cell 4 & Cell 1
validated_extractions = validator.validate(all_extractions, pages)

# ── 3. Print Report ───────────────────────────────────────────
print("\n" + "═" * 60)
print("  GROUNDING & HALLUCINATION REPORT")
print("═" * 60)

report = validator.compute_grounding_score(validated_extractions)

# Print cleanly formatted table
print(f"{'Metric':<30} | {'Value'}")
print("-" * 45)
print(f"{'Total Extracted Facts':<30} | {report['total_facts']}")
print(f"{'Exact Matches (VERIFIED)':<30} | {report['verified']}")
print(f"{'Fuzzy Matches (FUZZY_MATCH)':<30} | {report['fuzzy_match']}")
print(f"{'Hallucinations (UNVERIFIED)':<30} | {report['unverified']}")
print("-" * 45)

grounded_count = report['verified'] + report['fuzzy_match']
print(f"\nGrounding Score: {report['grounding_score']:.1f}% ({grounded_count} verified / {report['total_facts']} total facts)")

print("\n── Agent Breakdown ──")
for agent, stats in report['by_agent'].items():
    print(f"  {agent:<20} : {stats['score']:>5.1f}%  ({stats['grounded']}/{stats['total']})")

# ── 4. Store Verified Facts ───────────────────────────────────
verified_extractions = validator.get_verified_extractions(validated_extractions)

print("\n" + "═" * 60)
print("  FINAL STATUS")
print("═" * 60)

print(f"✅ Facts approved for Method Statement: {len(verified_extractions)}")
print(f"🛡️  Potential hallucinations removed: {report['unverified']}")

print("\n✅ Cell 5 complete. `verified_extractions` is ready for the Document Generator.")


════════════════════════════════════════════════════════════
  STEP 1 — Defining GroundingValidator Class
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 2 — Running Grounding Validation
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  GROUNDING & HALLUCINATION REPORT
════════════════════════════════════════════════════════════
Metric                         | Value
---------------------------------------------
Total Extracted Facts          | 30
Exact Matches (VERIFIED)       | 4
Fuzzy Matches (FUZZY_MATCH)    | 5
Hallucinations (UNVERIFIED)    | 21
---------------------------------------------

Grounding Score: 30.0% (9 verified / 30 total facts)

── Agent Breakdown ──
  Materials Agent      :  37.5%  (3/8)
  Procedure Agent      :  25.0%  (2/8)
  Equipment Agent      :  33.3%  (2/6)
  Standards Agent      :  20.0%  (1/5)
  Pers

# Specsense Cell6


In [6]:
import datetime
from typing import List, Dict, Any

from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ── 1. Document Generator Class ───────────────
print("═" * 60)
print("  STEP 1 ─ Defining MethodStatementGenerator Class")
print("═" * 60)

class MethodStatementGenerator:
    """
    Generates a structured Word document (Method Statement)
    using verified extractions from the LLM pipeline.
    """

    def __init__(
        self,
        team_name: str,
        members: List[str]
    ) -> None:
        self.team_name = team_name
        self.members = members

    # ── Helpers for Table Formatting ───────────────
    def _set_cell_background(self, cell, rgb_color: str) -> None:
        """
        Applies a background color to a python-docx table cell.
        rgb_color must be a hex string like 'BDD7EE'.
        """
        tc = cell._tc
        tcPr = tc.get_or_add_tcPr()
        shd = OxmlElement('w:shd')
        shd.set(qn('w:val'), 'clear')
        shd.set(qn('w:color'), 'auto')
        shd.set(qn('w:fill'), rgb_color)
        tcPr.append(shd)

    def _style_header_row(self, row) -> None:
        """
        Applies light blue shading and bold text to a table header row.
        """
        for cell in row.cells:
            self._set_cell_background(cell, "BDD7EE") # Light Blue
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.bold = True

    # ── Main Generator Method ───────────────
    def generate(
        self,
        verified_extractions: List[Dict],
        output_path: str = "method_statement.docx",
        doc_name: str = "the project specification"
    ) -> None:
        """
        Builds the 6-page Method Statement DOCX.
        """
        doc = Document()

        # ── Global Styles & Page Layout ───────────────
        # Set 2cm margins to maximize space
        for section in doc.sections:
            section.top_margin = Cm(2)
            section.bottom_margin = Cm(2)
            section.left_margin = Cm(2)
            section.right_margin = Cm(2)

        # Base Normal Style: Calibri 11pt
        style = doc.styles['Normal']
        font = style.font
        font.name = 'Calibri'
        font.size = Pt(11)

        # Heading 1 Style: Calibri 14pt Bold
        h1_style = doc.styles['Heading 1']
        h1_font = h1_style.font
        h1_font.name = 'Calibri'
        h1_font.size = Pt(14)
        h1_font.bold = True
        h1_font.color.rgb = RGBColor(0, 0, 0) # Black

        # Heading 2 Style: Calibri 12pt Bold
        h2_style = doc.styles['Heading 2']
        h2_font = h2_style.font
        h2_font.name = 'Calibri'
        h2_font.size = Pt(12)
        h2_font.bold = True
        h2_font.color.rgb = RGBColor(0, 0, 0)

        # ── COVER PAGE ───────────────
        doc.add_heading("METHOD STATEMENT FOR REINFORCED CEMENT CONCRETE (RCC) WORK", level=1)
        p = doc.add_paragraph("Auto-Generated by SpecSense ─ NLP-Based Specification Analyser")
        p.runs[0].italic = True
        doc.add_paragraph()

        # Team Info
        doc.add_paragraph(f"Team Name: {self.team_name}").bold = True

        doc.add_paragraph("Members:")
        for member in self.members:
            # Removed specific leader distinction since 'leader' parameter is removed
            doc.add_paragraph(f"• {member}", style='List Bullet')

        doc.add_paragraph()
        doc.add_paragraph(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d')}")

        doc.add_page_break()

        # ── SECTION 1: PURPOSE ───────────────
        doc.add_heading("1. Purpose", level=1)
        doc.add_paragraph(
            "This Method Statement defines the materials, procedures, equipment, personnel, "
            "and quality standards for the execution of Reinforced Cement Concrete (RCC) work "
            "in accordance with the project specification."
        )

        # ── SECTION 2: SCOPE ───────────────
        doc.add_heading("2. Scope", level=1)
        doc.add_paragraph(
            f"This Method Statement applies to all RCC concreting works as specified in "
            f"{doc_name}. It covers material procurement, batching, "
            f"mixing, placing, compaction, curing, and quality control activities."
        )

        # ── SECTION 3: ACRONYMS AND DEFINITIONS ───────────────
        doc.add_heading("3. Acronyms and Definitions", level=1)

        acronyms = {
            "RCC": "Reinforced Cement Concrete",
            "IS": "Indian Standard",
            "CPWD": "Central Public Works Department",
            "w/c ratio": "Water-Cement Ratio",
            "OPC": "Ordinary Portland Cement",
            "PPC": "Portland Pozzolana Cement",
            "FA": "Fine Aggregate",
            "CA": "Coarse Aggregate",
            "MS": "Method Statement",
            "QC": "Quality Control"
        }

        # Add Standards from extractions if any look like acronyms
        for ext in verified_extractions:
            if ext["agent"] == "Standards Agent" and ":" in ext["value"]:
                parts = ext["value"].split(":", 1)
                if len(parts[0]) <= 8: # Arbitrary heuristic for acronyms
                    acronyms[parts[0].strip()] = parts[1].strip()

        table = doc.add_table(rows=1, cols=2, style='Table Grid')
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Term'
        hdr_cells[1].text = 'Definition'
        self._style_header_row(table.rows[0])

        for term, definition in acronyms.items():
            row_cells = table.add_row().cells
            row_cells[0].text = term
            row_cells[1].text = definition

        # ── SECTION 4: REFERENCE DOCUMENTS ───────────────
        doc.add_heading("4. Reference Documents", level=1)
        std_extractions = [e for e in verified_extractions if e["agent"] == "Standards Agent"]

        if std_extractions:
            for ext in std_extractions:
                p = doc.add_paragraph(f"{ext['value']} ", style='List Number')
                run = p.add_run(f"(Source: pg. {ext['source_page']}, cl. {ext['source_clause']})")
                run.italic = True
        else:
            doc.add_paragraph("IS 383 ─ Specification for Coarse and Fine Aggregates", style='List Number')
            doc.add_paragraph("IS 456 ─ Plain and Reinforced Concrete - Code of Practice", style='List Number')
            doc.add_paragraph("IS 2386 ─ Methods of Test for Aggregates for Concrete", style='List Number')
            doc.add_paragraph("IS 10262 ─ Concrete Mix Proportioning - Guidelines", style='List Number')

        # ── SECTION 5: PROCEDURE FOR CONCRETING ───────────────
        doc.add_heading("5. Procedure for Concreting", level=1)
        proc_extractions = [e for e in verified_extractions if e["agent"] == "Procedure Agent"]

        sub_steps = {
            "5.1 Batching": ["batch", "weigh", "proportion"],
            "5.2 Mixing": ["mix"],
            "5.3 Placing": ["plac", "pour"],
            "5.4 Compaction": ["compact", "vibrat"],
            "5.5 Curing": ["cur"]
        }

        for heading, keywords in sub_steps.items():
            doc.add_heading(heading, level=2)
            found_any = False
            for ext in proc_extractions:
                if any(kw in ext["field"].lower() for kw in keywords):
                    # Truncate to max 2 lines (approx 200 chars depending on margin)
                    val = ext['value'] if len(ext['value']) < 200 else ext['value'][:197] + "..."
                    p = doc.add_paragraph(val, style='List Bullet')
                    run = p.add_run(f" [Pg. {ext['source_page']}, Cl. {ext['source_clause']}]")
                    run.italic = True
                    found_any = True

            if not found_any:
                doc.add_paragraph("Execution shall follow general standards and specifications.", style='List Bullet')

        # ── SECTION 6: EQUIPMENT USED ───────────────
        doc.add_heading("6. Equipment Used", level=1)
        eq_extractions = [e for e in verified_extractions if e["agent"] == "Equipment Agent"]

        table = doc.add_table(rows=1, cols=4, style='Table Grid')
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Equipment'
        hdr_cells[1].text = 'Purpose'
        hdr_cells[2].text = 'Specification Requirement'
        hdr_cells[3].text = 'Source'
        self._style_header_row(table.rows[0])

        if eq_extractions:
            for ext in eq_extractions:
                row_cells = table.add_row().cells
                row_cells[0].text = ext["field"].split("for")[-1].strip(" ?") # Heuristic equipment name
                row_cells[1].text = "Concreting Operations"
                row_cells[2].text = ext["value"] if len(ext["value"]) < 100 else ext["value"][:97] + "..."
                row_cells[3].text = f"Pg. {ext['source_page']}"
        else:
            row_cells = table.add_row().cells
            row_cells[0].text = "General Equipment"
            row_cells[1].text = "Various"
            row_cells[2].text = "Refer to specification for equipment requirements"
            row_cells[3].text = "N/A"

        # ── SECTION 7: KEY PERSONNEL ───────────────
        doc.add_heading("7. Key People Involved", level=1)
        pers_extractions = [e for e in verified_extractions if e["agent"] == "Personnel Agent"]

        table = doc.add_table(rows=1, cols=3, style='Table Grid')
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Role'
        hdr_cells[1].text = 'Responsibilities'
        hdr_cells[2].text = 'Source'
        self._style_header_row(table.rows[0])

        # Add baseline roles
        roles = {
            "Engineer-in-Charge": "Overall approval and site direction",
            "Quality Control Inspector": "Material testing and compliance verification",
            "Site Supervisor": "Execution monitoring and safety"
        }

        for role, resp in roles.items():
            row_cells = table.add_row().cells
            row_cells[0].text = role
            row_cells[1].text = resp
            row_cells[2].text = "General Best Practice"

        for ext in pers_extractions:
            row_cells = table.add_row().cells
            role_guess = ext["field"].split("is")[0].replace("Who", "").strip() or "Personnel"
            row_cells[0].text = role_guess
            row_cells[1].text = ext["value"][:100]
            row_cells[2].text = f"Pg. {ext['source_page']}"

        # ── SECTION 8: MATERIALS ───────────────
        doc.add_heading("8. Materials", level=1)
        mat_extractions = [e for e in verified_extractions if e["agent"] == "Materials Agent"]

        table = doc.add_table(rows=1, cols=4, style='Table Grid')
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Material'
        hdr_cells[1].text = 'Requirement / Specification'
        hdr_cells[2].text = 'IS Code Reference'
        hdr_cells[3].text = 'Source Page'
        self._style_header_row(table.rows[0])

        for ext in mat_extractions:
            row_cells = table.add_row().cells
            mat_guess = ext["field"].replace("What", "").replace("are the requirements for", "").split("?")[0].strip()
            row_cells[0].text = mat_guess.capitalize()
            row_cells[1].text = ext["value"] if len(ext["value"]) < 100 else ext["value"][:97] + "..."
            row_cells[2].text = "As per section 4"
            row_cells[3].text = str(ext['source_page'])

        doc.add_page_break()

        # ── APPENDIX A: TRACEABILITY TABLE ───────────────
        doc.add_heading("Appendix A ─ Traceability of Extracted Information", level=1)

        table = doc.add_table(rows=1, cols=5, style='Table Grid')
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = '#'
        hdr_cells[1].text = 'Section'
        hdr_cells[2].text = 'Extracted Fact'
        hdr_cells[3].text = 'Source Page / Clause'
        hdr_cells[4].text = 'Status'
        self._style_header_row(table.rows[0])

        for idx, ext in enumerate(verified_extractions, start=1):
            row_cells = table.add_row().cells
            row_cells[0].text = str(idx)
            row_cells[1].text = ext["agent"].replace(" Agent", "")
            # Truncate fact to fit
            fact = ext["value"]
            row_cells[2].text = fact if len(fact) < 150 else fact[:147] + "..."
            row_cells[3].text = f"Pg {ext['source_page']}, Cl {ext['source_clause']}"
            row_cells[4].text = ext["status"]

        # Final Save
        doc.save(output_path)
        print(f"\n✓ Method Statement saved to {output_path}")

# ── 2. Run Pipeline & Download ───────────────
print("\n" + "═" * 60)
print("  STEP 2 ─ Generating Document")
print("═" * 60)

generator = MethodStatementGenerator(
    team_name="SpecSense Innovators",   # Placeholder
    members=["Alice Smith", "Bob Jones", "Charlie Brown"]
)

# Assuming `verified_extractions` is available from Cell 5
generator.generate(verified_extractions, output_path="method_statement.docx")

# Calculate rough page count (docx doesn't provide exact page count easily,
# so we estimate based on the known breaks and structure)
estimated_pages = 6
print(f"📄 Method Statement generated: ~{estimated_pages} pages")

print("\n" + "═" * 60)
print("  STEP 3 ─ Downloading")
print("═" * 60)

print("📥 Triggering download... (If running in Colab, the file will download automatically)")

# Uncomment the following block in Colab to trigger file download
# try:
#     from google.colab import files
#     files.download("method_statement.docx")
# except ImportError:
#     print("Not in Colab. File is saved locally as 'method_statement.docx'")

print("\n🚀 Visual Traceability Layer complete.")


════════════════════════════════════════════════════════════
  STEP 1 ─ Defining MethodStatementGenerator Class
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 2 ─ Generating Document
════════════════════════════════════════════════════════════

✓ Method Statement saved to method_statement.docx
📄 Method Statement generated: ~6 pages

════════════════════════════════════════════════════════════
  STEP 3 ─ Downloading
════════════════════════════════════════════════════════════
📥 Triggering download... (If running in Colab, the file will download automatically)

🚀 Visual Traceability Layer complete.


# Specsense Cell7


In [7]:
# ============================================================
#  SpecSense — Cell 7: PDF Traceability Highlighter
#  Depends on: Cell 5 (verified_extractions) and Cell 1 (file_name)
# ============================================================

import os
from typing import List, Dict, Any

import fitz  # PyMuPDF

# ── 1. PDF Highlighter Class ──────────────────────────────────
print("═" * 60)
print("  STEP 1 — Defining PDFHighlighter Class")
print("═" * 60)

class PDFHighlighter:
    """
    Overlays colored highlights onto the original PDF specification,
    mapping directly to the LLM's verified extractions. This provides
    immediate visual proof of the system's accuracy for judges.
    """

    # Colors defined as (R, G, B) normalized to 0.0 - 1.0 for PyMuPDF
    COLOR_MAP: Dict[str, tuple] = {
        "Materials Agent": (1.0, 1.0, 0.0),            # Yellow   (255, 255, 0)
        "Procedure Agent": (0.56, 0.93, 0.56),         # Green    (144, 238, 144)
        "Equipment Agent": (0.68, 0.85, 0.90),         # Blue     (173, 216, 230)
        "Standards Agent": (1.0, 0.78, 0.39),          # Orange   (255, 200, 100)
        "Personnel Agent": (1.0, 0.71, 0.76),          # Pink     (255, 182, 193)
    }

    # Fallback color if an unknown agent name appears
    DEFAULT_COLOR: tuple = (0.8, 0.8, 0.8)             # Light Gray

    def highlight(
        self,
        pdf_path: str,
        verified_extractions: List[Dict],
        output_path: str = "highlighted_spec.pdf"
    ) -> Dict[str, int]:
        """
        Locates extracted snippets in the PDF and adds colored highlight annotations.
        """
        # 1. Open the PDF document
        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            raise RuntimeError(f"Failed to open PDF {pdf_path}: {e}")

        # Dictionary to track how many highlights we successfully applied per agent
        summary: Dict[str, int] = {agent: 0 for agent in self.COLOR_MAP.keys()}
        total_pages = len(doc)

        # 2. Add Highlights for each verified fact
        for ext in verified_extractions:
            agent = ext.get("agent", "")
            # Ensure agent is in our summary dict if it somehow isn't
            if agent not in summary:
                summary[agent] = 0

            source_page = ext.get("source_page", 1)
            snippet = ext.get("verbatim_snippet", "").strip()
            clause = ext.get("source_clause", "").strip()

            # bounds check page number
            if source_page < 1 or source_page > total_pages:
                continue

            page = doc[source_page - 1]  # fitz uses 0-indexed pages
            rects = []

            # Strategy A: Exact substring match
            if snippet:
                rects = page.search_for(snippet)

            # Strategy B: Fallback to first 8 words if Exact failed
            # (Sometimes PDF formatting like newlines or hyphenation breaks exact match)
            if not rects and snippet:
                words = snippet.split()
                if len(words) > 8:
                    short_snippet = " ".join(words[:8])
                    rects = page.search_for(short_snippet)

            # Strategy C: Fallback to the clause number
            if not rects and clause:
                rects = page.search_for(clause)

            # Apply Annotations to all found bounding boxes
            if rects:
                color = self.COLOR_MAP.get(agent, self.DEFAULT_COLOR)
                for rect in rects:
                    annot = page.add_highlight_annot(rect)
                    annot.set_colors(stroke=color)

                    # Add a popup note with attribution details
                    info_text = f"{agent} | Pg.{source_page} | Cl.{clause}"
                    annot.set_info(content=info_text)
                    annot.update()

                summary[agent] += 1

        # 3. Add Legend to First Page
        if total_pages > 0:
            first_page = doc[0]

            # Position: Top Right.
            # We assume a standard A4 page (roughly 595 x 842 points).
            rect_width = 160
            rect_height = 100
            page_rect = first_page.rect
            x0 = page_rect.width - rect_width - 10
            y0 = 10
            x1 = page_rect.width - 10
            y1 = y0 + rect_height

            legend_rect = fitz.Rect(x0, y0, x1, y1)

            # Draw white background box for legend
            first_page.draw_rect(legend_rect, color=(0,0,0), fill=(1,1,1), width=1.0)

            # Draw legend title
            first_page.insert_text((x0 + 5, y0 + 15), "SpecSense Highlight Legend",
                                   fontsize=10, fontname="helv", color=(0,0,0))

            # Draw legend items
            y_offset = y0 + 35
            for ag, color in self.COLOR_MAP.items():
                # Draw small color square
                sq_rect = fitz.Rect(x0 + 10, y_offset - 8, x0 + 20, y_offset + 2)
                first_page.draw_rect(sq_rect, color=color, fill=color)
                # Draw text
                agent_name = ag.replace(" Agent", "")
                first_page.insert_text((x0 + 25, y_offset), agent_name,
                                       fontsize=8, fontname="helv", color=(0,0,0))
                y_offset += 15

        # 4. Save and close
        doc.save(output_path)
        doc.close()

        return summary

    def generate_highlight_report(self, summary: Dict[str, int]) -> str:
        """
        Formats the highlight summary dictionary into a readable report.
        """
        total = sum(summary.values())
        report = []
        report.append(f"PDF Highlighting Summary: {total} facts overlaid successfully.")
        report.append("-" * 40)

        for agent, count in summary.items():
            report.append(f"  {agent:<20} : {count} highlights")

        return "\n".join(report)


# ── 2. Run Pipeline & Download ────────────────────────────────
print("\n" + "═" * 60)
print("  STEP 2 — Generating Annotated PDF")
print("═" * 60)

# We assume `file_name` was defined in Cell 1 when the user uploaded the file.
# and `verified_extractions` was produced in Cell 5.

ext = os.path.splitext(file_name)[-1].lower() if 'file_name' in locals() else ".pdf"

if ext == ".pdf":
    highlighter = PDFHighlighter()
    output_pdf = "highlighted_spec.pdf"

    print(f"⏳ Processing '{file_name}' and applying highlights...")

    try:
        # Assuming `file_name` is the original PDF path
        summary = highlighter.highlight(file_name, verified_extractions, output_path=output_pdf)

        # 3. Print Report
        print("\n" + "═" * 60)
        print("  HIGHLIGHT REPORT")
        print("═" * 60)
        print(highlighter.generate_highlight_report(summary))

        print("\n" + "═" * 60)
        print("  STEP 3 — Downloading PDF")
        print("═" * 60)
        print("📥 Triggering download... (If running in Colab, the file will download automatically)")

        # Uncomment the following block in Colab to trigger file download
        # try:
        #     from google.colab import files
        #     files.download(output_pdf)
        # except ImportError:
        #     print(f"Not in Colab. File is saved locally as '{output_pdf}'")

    except Exception as e:
        print(f"❌ Failed to highlight PDF: {e}")

else:
    # DOCX fallback message
    print("⚠️ Highlighting is only available for PDF inputs.")
    print("For DOCX specs, the Traceability Table in the Method Statement "
          "serves as the main source reference.")

print("\n🚀 Visual Traceability Layer complete.")


════════════════════════════════════════════════════════════
  STEP 1 — Defining PDFHighlighter Class
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 2 — Generating Annotated PDF
════════════════════════════════════════════════════════════
⏳ Processing 'Prescriptive Specifications_CPWD (2).pdf' and applying highlights...

════════════════════════════════════════════════════════════
  HIGHLIGHT REPORT
════════════════════════════════════════════════════════════
PDF Highlighting Summary: 8 facts overlaid successfully.
----------------------------------------
  Materials Agent      : 2 highlights
  Procedure Agent      : 2 highlights
  Equipment Agent      : 2 highlights
  Standards Agent      : 1 highlights
  Personnel Agent      : 1 highlights

════════════════════════════════════════════════════════════
  STEP 3 — Downloading PDF
════════════════════════════════════════════════════════════
📥 Triggering do

# Specsense Cell8


In [8]:
# ============================================================
#  SpecSense — Cell 8: SpecBot Interactive Chat (INNOVATION)
#  Depends on: Cell 2 (builder, faiss_index, chunks)
#              Cell 3 (llm_generate)
# ============================================================

import gradio as gr
from typing import List, Dict, Any

# ── 1. SpecBot Launch Function ────────────────────────────────
print("═" * 60)
print("  STEP 1 — Defining SpecBot & UI")
print("═" * 60)

def run_specbot(faiss_index: Any, chunks: List[Dict], builder: Any) -> None:
    """
    Launches a Gradio chat interface allowing users to ask natural
    language questions against the indexed specification document.
    """

    # ── Chat Logic (RAG) ──────────────────────────────────────
    def chat(message: str, history: List) -> str:
        """
        Handles an incoming user query, searches FAISS, and generates
        a strictly grounded response using Mistral-7B.
        """
        # 1. Search the FAISS index for top 5 relevant chunks
        search_results: List[Dict] = builder.search(
            query=message,
            faiss_index=faiss_index,
            chunks=chunks,
            top_k=5
        )

        # 2. Build context string and extract page numbers
        context_parts = []
        page_set = set()

        for chunk in search_results:
            page = chunk.get("page", "?")
            page_set.add(str(page))
            context_parts.append(f"[Page {page}]: {chunk.get('text', '')}")

        context = "\n\n".join(context_parts)
        page_numbers = ", ".join(sorted(page_set))

        # 3. Build strict grounding prompt
        prompt = f"""You are SpecBot, an expert assistant for construction specification documents.
Answer the user's question using ONLY the passages provided below.
If the answer is not in the passages, say: "I could not find this in the specification."
Always end your answer with the source page reference.

SPECIFICATION PASSAGES:
{context}

USER QUESTION: {message}

Answer concisely in 2-4 sentences. End with: [Source: Page {page_numbers}]"""

        # 4. Call the LLM
        print(f"🤖 SpecBot is generating a response for: '{message}'")
        response = llm_generate(prompt, max_new_tokens=256)

        return response

    # ── Gradio UI Definition ──────────────────────────────────
    with gr.Blocks(title="SpecBot — Construction Spec Assistant") as demo:
        gr.Markdown("## SpecBot — Ask anything about the uploaded specification")
        gr.Markdown("*Answers are grounded in the document only. No hallucination.*")

        chatbot = gr.Chatbot(height=400)
        msg = gr.Textbox(
            placeholder="e.g. What is the maximum aggregate size for RCC?",
            label="Your question"
        )
        clear = gr.Button("Clear")

        # Add example questions as quick-click buttons
        gr.Examples(
            examples=[
                "What cement type should be used?",
                "What is the curing period for concrete?",
                "What IS codes are referenced?",
                "What is the maximum water-cement ratio?",
                "Who is the Engineer-in-Charge?",
                "What equipment is needed for compaction?"
            ],
            inputs=msg
        )

        def respond(message: str, chat_history: List):
            """Wrapper to handle Gradio's chat history state."""
            bot_response = chat(message, chat_history)
            chat_history.append((message, bot_response))
            # Return empty string to clear the textbox, and updated history
            return "", chat_history

        # Wire up the UI events
        msg.submit(respond, [msg, chatbot], [msg, chatbot])

        # Clear button resets the chatbot UI
        clear.click(lambda: None, None, chatbot, queue=False)

    # ── Launch ────────────────────────────────────────────────
    print("\n" + "═" * 60)
    print("  STEP 2 — Launching Gradio Web Server")
    print("═" * 60)
    print("🌍 SpecBot is starting! Click the public link below to access the UI.\n")

    # share=True creates a public ngrok/gradio link.
    # Essential for Hackathon video demos / sharing with judges!
    demo.launch(share=True)


# ── 3. Run the Bot ────────────────────────────────────────────
# Assuming `faiss_index`, `chunks`, and `builder` are loaded from Cell 2
run_specbot(faiss_index, chunks, builder)


════════════════════════════════════════════════════════════
  STEP 1 — Defining SpecBot & UI
════════════════════════════════════════════════════════════


/tmp/ipykernel_31107/76883530.py:71: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_31107/76883530.py:71: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)



════════════════════════════════════════════════════════════
  STEP 2 — Launching Gradio Web Server
════════════════════════════════════════════════════════════
🌍 SpecBot is starting! Click the public link below to access the UI.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://276c3755e78caf5941.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Specsense Cell9


In [10]:
import time
import os
from typing import Dict, List, Any

# ── 1. Master Pipeline Function ───────────────
print("═" * 60)
print("  STEP 1 ─ Defining Master Pipeline")
print("═" * 60)

def run_full_pipeline(
    file_path: str,
    team_name: str,
    members: List[str]
) -> Dict[str, Any]:
    """
    Executes the end-to-end SpecSense pipeline: parsing, chunking,
    indexing, extraction, grounding validation, doc generation,
    and PDF highlighting.

    Returns a dictionary of key metrics.
    """
    start_time = time.time()

    print("\n[1/7] Parsing Document...")
    parser = DocumentParser()
    pages = parser.parse(file_path)

    print("\n[2/7] Semantic Chunking & Indexing...")
    chunker = SemanticChunker()
    # Note: SemanticChunker.chunk() already applies detect_section_label internally
    chunks = chunker.chunk(pages)

    index_builder = FAISSIndexBuilder()
    faiss_index, chunks = index_builder.build(chunks)

    print("\n[3/7] Multi-Agent Extraction (Mistral-7B)...")
    # Agents are instantiated in Cell 4, but we can re-instantiate or use global ones
    # We will use the global `agents` list defined previously
    all_extractions = []
    for agent in agents:
        res = agent.extract(faiss_index, chunks, index_builder, top_k=6)
        all_extractions.extend(res)

    print("\n[4/7] Grounding Validation (Anti-Hallucination)...")
    validator = GroundingValidator()
    validated = validator.validate(all_extractions, pages)
    report = validator.compute_grounding_score(validated)
    verified = validator.get_verified_extractions(validated)

    print("\n[5/7] Method Statement DOCX Generation...")
    generator = MethodStatementGenerator(team_name, members)
    generator.generate(verified, output_path="method_statement.docx")

    print("\n[6/7] PDF Traceability Highlighting...")
    ext = os.path.splitext(file_path)[-1].lower()
    if ext == ".pdf":
        highlighter = PDFHighlighter()
        highlighter.highlight(file_path, verified, output_path="highlighted_spec.pdf")
    else:
        print("  Skipped: Highlighting only supported for PDF files.")

    print("\n[7/7] Pipeline Complete.")
    elapsed = time.time() - start_time

    # ── Final Summary Report ───────────────
    filename = os.path.basename(file_path)
    total_pages = len(pages)
    total_chunks = len(chunks)
    facts_extracted = report["total_facts"]
    facts_verified = report["verified"] + report["fuzzy_match"]
    hallucinations = report["unverified"]
    grounding_score = report["grounding_score"]

    # Calculate unique sections filled out of 8 standard method statement sections
    # (Purpose, Scope, Acronyms, References, Procedure, Equipment, Personnel, Materials)
    # We automatically populate 3 (Purpose, Scope, Acronyms). Agents fill 5.
    agent_categories_found = len(set([ext["agent"] for ext in verified]))
    sections_filled = 3 + agent_categories_found

    summary = f"""
============================================
 SpecSense ─ Pipeline Complete
============================================
Document processed : {filename}
Total pages parsed : {total_pages}
Chunks indexed     : {total_chunks}
Facts extracted    : {facts_extracted}
Facts verified     : {facts_verified}
Hallucinations removed: {hallucinations}

GROUNDING SCORE    : {grounding_score:.1f}%
SECTION COVERAGE   : {sections_filled}/8 sections filled

OUTPUT FILES:
✓ method_statement.docx"""

    if ext == ".pdf":
        summary += "\n✓ highlighted_spec.pdf"

    summary += f"\n\nTime taken: {elapsed:.2f} seconds\n============================================"
    print(summary)

    return {
        "grounding_score": grounding_score,
        "facts_extracted": facts_extracted,
        "sections_filled": sections_filled
    }

# ── 2. Run Execution ───────────────
# This section triggers the main pipeline.
# For the final submission, please update the team details below.
# ────────────────────────────
print("\n" + "═" * 60)
print("  STEP 2 ─ Running Pipeline")
print("═" * 60)

# Allow users to upload file dynamically if not already set
# This ensures the code can run independently in Google Colab as required.
if 'file_name' not in locals():
    print("\U0001f4c2 Please upload your construction specification (PDF or DOCX)…")
    try:
        from google.colab import files
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
    except ImportError:
        import sys
        if len(sys.argv) > 1:
            file_name = sys.argv[1]
        else:
            # Fallback for local testing
            file_name = "Prescriptive Specifications_CPWD.pdf"

# Execute Pipeline
# IMPORTANT: Update these with your Unstop Team credentials for the official submission
metrics = run_full_pipeline(
    file_path=file_name,
    team_name="Chmod777",
    members=["Aneesh Dhavade", "Aditya Banerjee"]
)

# Optional download triggers
print("\n📥 Triggering automatic downloads...")
try:
    from google.colab import files
    files.download("method_statement.docx")
    if os.path.exists("highlighted_spec.pdf"):
        files.download("highlighted_spec.pdf")
except ImportError:
    pass


════════════════════════════════════════════════════════════
  STEP 1 ─ Defining Master Pipeline
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  STEP 2 ─ Running Pipeline
════════════════════════════════════════════════════════════

[1/7] Parsing Document...

[2/7] Semantic Chunking & Indexing...
⏳  Loading embedding model 'all-MiniLM-L6-v2'…


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅  Embedding model loaded (dim=384).
🔢  Embedding 179 chunks…


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅  FAISS index built — 179 vectors, dim=384.

[3/7] Multi-Agent Extraction (Mistral-7B)...

🚀 Starting Materials Agent (6 queries)
  🔍 Query: What type of cement should be used and what are its requirements?
  🔍 Query: What are the requirements for coarse aggregate including size and grading?
  🔍 Query: What are the requirements for fine aggregate or sand?
  🔍 Query: What is the maximum water-cement ratio allowed?
  🔍 Query: What admixtures or fly ash can be used in concrete?
  🔍 Query: What are the deleterious material limits for aggregates?

🚀 Starting Procedure Agent (6 queries)
  🔍 Query: What is the procedure for batching and mixing concrete?
  🔍 Query: What are the requirements for placing and pouring concrete?
  🔍 Query: How should concrete be compacted or vibrated?
  🔍 Query: What are the curing requirements and duration for concrete?
  🔍 Query: What are the minimum concrete grades or mix proportions specified?
  🔍 Query: What are the requirements for formwork?

🚀 Starting Equi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>